**EDA SVC MODEL**

This notebook shows the inital application and exploration of the SVC model at one iteration of the labeling stage.

In [14]:
import pandas as pd
import numpy as np
import regex as re
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import textwrap
from sklearn.preprocessing import OneHotEncoder
import string
from sklearn.model_selection import cross_val_score


nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /Users/emma/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
labeled = pd.read_csv('labeled_data_clean.csv')
labeled.shape

(1492, 11)

In [16]:
labeled = labeled[labeled['my_label'] != 'unclear'].reset_index(drop=True)
labeled['my_label'].value_counts()

my_label
no stance    580
prochoice    479
prolife      409
Name: count, dtype: int64

In [17]:
# regex replacing punc
labeled['body'] = labeled['body'].str.replace(f"[{string.punctuation}]", " ", regex=True).str.strip()

In [18]:
# encoding labels
label_encoder = LabelEncoder()
y  = label_encoder.fit_transform(labeled['my_label'])
(dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))

{'no stance': 0, 'prochoice': 1, 'prolife': 2}

In [19]:
# tf idf
tfidf = TfidfVectorizer(max_features=5000,
                        stop_words='english'
                        )
X  = tfidf.fit_transform(labeled['body'])

print('training shape:', X.shape)

training shape: (1468, 5000)


In [20]:
word_importance = np.asarray(X.sum(axis=0)).flatten()

fn = tfidf.get_feature_names_out()

top = np.argsort(word_importance)[::-1][:10]

for i in top:
    print(f"{fn[i]}: {word_importance[i]}")


abortion: 48.62919754893688
life: 47.782214930474126
right: 46.94476169240697
rights: 43.74013396616526
people: 38.04101100687219
women: 34.06695609233841
autonomy: 33.45626312136368
pro: 31.89958904148394
bodily: 30.623161542244052
just: 30.5833520840149


- this interesting, bc abortion is so high. I would expect (in a dataset with comments that should all be pertaining to abortion) that this would have a lower idf score. Also rights is interesting bc its used by both sides. 

In [21]:
# labeled['body'].str.contains('abortion', case=False, na=False).mean()

# 32% interesting..

In [22]:
# train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=13
)


# SVC

In [23]:
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

def run_svm(X_train, y_train, X_test, y_test):
    
    svm_model = SVC(kernel="linear", 
                    C=1.0, 
                    probability=True, 
                    class_weight="balanced")

    svm_model.fit(X_train, y_train)

    y_pred = svm_model.predict(X_test)

    print('acc:', accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))
    print("")
    print(confusion_matrix(y_test, y_pred))

    return svm_model

reg_svm_model = run_svm(X_train, y_train, X_test, y_test)

acc: 0.717687074829932
              precision    recall  f1-score   support

   no stance       0.72      0.80      0.76       110
   prochoice       0.62      0.65      0.63        93
     prolife       0.84      0.69      0.76        91

    accuracy                           0.72       294
   macro avg       0.73      0.71      0.72       294
weighted avg       0.73      0.72      0.72       294


[[88 20  2]
 [23 60 10]
 [11 17 63]]


In [24]:
def get_missclassified(y_test, y_pred, X_test_indices, n):
    y_test = np.array(y_test)
    y_pred = np.array(y_pred)

    misclassified = np.where(y_test != y_pred)[0]

    for i in misclassified[:n]:
        original_index = X_test_indices[i]
        print(f"True Label: {label_encoder.classes_[y_test[i]]}")
        print(f"Predicted Label: {label_encoder.classes_[y_pred[i]]}")
        text = labeled.loc[original_index, 'body']
        wrapped_text = textwrap.fill(text, width=80)
        print(wrapped_text)
        print('--' * 50)
        print("")
        

In [25]:
X_train_indices, X_test_indices, y_train, y_test = train_test_split(
    labeled.index, y, test_size=0.2, random_state=13, stratify=y
)

y_pred = reg_svm_model.predict(X_test)
get_missclassified(y_test, y_pred, X_test_indices, 10)

True Label: prochoice
Predicted Label: no stance
sounds like gop is surely but slowly turning back the time to 1800s where
certain population had no civil rights    curtail womens rights check curtail
rights for gays  lesbians  trans check curtail voting rights for some check
next on the agenda  curtail rights for blacks and asians   america in 2024 will
look a lot like 1804
----------------------------------------------------------------------------------------------------

True Label: no stance
Predicted Label: prolife
98  of catholics  the ones who were up in arms about this bill  use birth
control  they literally have nothing to argue about as they all use birth
control  and they don t care  fuckin nitwits
----------------------------------------------------------------------------------------------------

True Label: prochoice
Predicted Label: prolife
no  you don t seem to get my logic  because that s not what i said  my point is
that we all pay into a big pot and some of the mone

- doesnt seem to be a pattern

In [26]:
# from sklearn.pipeline import Pipeline
# from sklearn.model_selection import GridSearchCV

# pipeline = Pipeline(
#     [
#         ("tfidf", TfidfVectorizer(ngram_range=(1,2))), 
#         ("clf", SVC(kernel="linear", probability=True, class_weight="balanced")),
#     ]
# )


In [27]:
# X  = labeled['body']

# # train test split
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, random_state=13
# )


In [28]:
# grid = {
#     "tfidf__ngram_range": [(1,1), (1,2), (1,3)], 
#     "clf__C": [0.1, 1, 3],
#     "tfidf__stop_words": [None,'english'],
#     "clf__kernel": ['linear', 'rbf'],
# }

# grid_search = GridSearchCV(pipeline, grid, cv=5, scoring="accuracy", n_jobs=-1)

# grid_search.fit(X_train, y_train)

# "Best Parameters:", grid_search.best_params_
# "Best Accuracy Score:", grid_search.best_score_


# applying those

In [29]:
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

# tf idf
tfidf = TfidfVectorizer(max_features=5000, 
                        ngram_range = (1, 2), 
                        stop_words='english')
X  = tfidf.fit_transform(labeled['body'])

# train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=13
)

grid_svm = run_svm(X_train, y_train, X_test, y_test)

acc: 0.7721088435374149
              precision    recall  f1-score   support

   no stance       0.78      0.81      0.80       116
   prochoice       0.72      0.69      0.70        96
     prolife       0.82      0.82      0.82        82

    accuracy                           0.77       294
   macro avg       0.77      0.77      0.77       294
weighted avg       0.77      0.77      0.77       294


[[94 15  7]
 [22 66  8]
 [ 4 11 67]]


In [30]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

def cross_val(X_train, y_train):

    strat_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=13)
    svm_model_cv = SVC(kernel="linear", C=1.0, probability=True, class_weight="balanced")

    cv_acc = cross_val_score(svm_model_cv, X_train, y_train, cv=strat_kfold, scoring="accuracy")
    cv_f1 = cross_val_score(svm_model_cv, X_train, y_train, cv=strat_kfold, scoring="f1_macro")

    print("cv scores:", cv_acc)
    print(('avg acc:', cv_acc.mean()))
    print("std acc:", cv_acc.std())
    print('avg f1', cv_f1.mean())

cross_val(X_train, y_train)

cv scores: [0.7106383  0.7787234  0.74042553 0.7106383  0.7008547 ]
('avg acc:', 0.728256046553919)
std acc: 0.028518956680266678
avg f1 0.7283629160566647


In [31]:
X_train_indices, X_test_indices, y_train, y_test = train_test_split(
    labeled.index, y, test_size=0.2, random_state=13, stratify=y
)

y_pred = reg_svm_model.predict(X_test)
get_missclassified(y_test, y_pred, X_test_indices, 20)

True Label: prochoice
Predicted Label: no stance
anti choice
----------------------------------------------------------------------------------------------------

True Label: prochoice
Predicted Label: no stance
sounds like gop is surely but slowly turning back the time to 1800s where
certain population had no civil rights    curtail womens rights check curtail
rights for gays  lesbians  trans check curtail voting rights for some check
next on the agenda  curtail rights for blacks and asians   america in 2024 will
look a lot like 1804
----------------------------------------------------------------------------------------------------

True Label: prochoice
Predicted Label: no stance
im morally pro life  i just believe a womans right to bodily autonomy supersedes
the fetuss right to life   go back 250 years   i m morally anti slavery  i just
believe a slave owner s right to their property supersedes the black s right to
freedom    no difference  you re fine with certain humans being tre

# testing w/ subreddit as feature

In [32]:
encoder = OneHotEncoder(handle_unknown="ignore")
subreddit_code = encoder.fit_transform(labeled[['subreddit']]).toarray()

subreddits = pd.DataFrame(subreddit_code, columns=encoder.get_feature_names_out(["subreddit"]))
subreddits

# 1 for pol, 0 for con

,subreddit_Conservative,subreddit_politics
0,0.0,1.0
1,0.0,1.0
2,0.0,1.0
3,0.0,1.0
4,0.0,1.0
...,...,...
1463,1.0,0.0
1464,1.0,0.0
1465,1.0,0.0
1466,1.0,0.0


In [33]:
import scipy.sparse as sp

X_text = tfidf.fit_transform(labeled['body']) 
X_combined = sp.hstack((X_text, subreddits))

In [34]:
X_train, X_test, y_train, y_test = train_test_split(
    X_combined, y, test_size=0.2, random_state=13, stratify=y
)

svm_model_subreddits = run_svm(X_train, y_train, X_test, y_test)

acc: 0.7380952380952381
              precision    recall  f1-score   support

   no stance       0.74      0.76      0.75       116
   prochoice       0.77      0.58      0.66        96
     prolife       0.72      0.89      0.79        82

    accuracy                           0.74       294
   macro avg       0.74      0.74      0.74       294
weighted avg       0.74      0.74      0.73       294


[[88 14 14]
 [25 56 15]
 [ 6  3 73]]


In [35]:
cross_val(X_train, y_train)

cv scores: [0.71914894 0.76170213 0.73191489 0.74468085 0.72222222]
('avg acc:', 0.7359338061465721)
std acc: 0.01566147017453944
avg f1 0.7357930965969091


In [36]:
# avg 75% accuray when subreddits are in

In [37]:
X_train_indices, X_test_indices, y_train, y_test = train_test_split(
    labeled.index, y, test_size=0.2, random_state=13, stratify=y
)

y_pred = svm_model_subreddits.predict(X_test)
get_missclassified(y_test, y_pred, X_test_indices, 10)

True Label: prochoice
Predicted Label: no stance
sounds like gop is surely but slowly turning back the time to 1800s where
certain population had no civil rights    curtail womens rights check curtail
rights for gays  lesbians  trans check curtail voting rights for some check
next on the agenda  curtail rights for blacks and asians   america in 2024 will
look a lot like 1804
----------------------------------------------------------------------------------------------------

True Label: prochoice
Predicted Label: prolife
im morally pro life  i just believe a womans right to bodily autonomy supersedes
the fetuss right to life   go back 250 years   i m morally anti slavery  i just
believe a slave owner s right to their property supersedes the black s right to
freedom    no difference  you re fine with certain humans being treated as sub
human  but you wouldn t dare do it yourself  how morally righteous you are
------------------------------------------------------------------------------

- doesnt seem to be an immediately obvious pattern

# score

In [38]:
# interested about possibly adding in score..

labels = ['no stance', 'prochoice', 'prolife']

for l in labels:
    print(l)
    df = labeled[labeled['my_label'] == l]
    print(df['score'].describe())
    print('--'*10)

no stance
count    580.000000
mean       5.860345
std       21.554834
min     -272.000000
25%        1.000000
50%        2.000000
75%        6.000000
max      149.000000
Name: score, dtype: float64
--------------------
prochoice
count    479.000000
mean       8.014614
std       23.543732
min      -29.000000
25%        1.000000
50%        2.000000
75%        7.000000
max      333.000000
Name: score, dtype: float64
--------------------
prolife
count    409.000000
mean       4.300733
std       13.356170
min      -28.000000
25%        0.000000
50%        2.000000
75%        7.000000
max      140.000000
Name: score, dtype: float64
--------------------


In [39]:
from sklearn.preprocessing import StandardScaler
import scipy.sparse as sp

In [40]:
scaler = StandardScaler()
score = scaler.fit_transform(labeled[['score']])

In [41]:
tfidf = TfidfVectorizer(max_features=5000, 
                        ngram_range=(1, 2), 
                        stop_words='english')
X_tfidf = tfidf.fit_transform(labeled['body'])

X_combined = sp.hstack((X_tfidf, score))


In [42]:
X_train, X_test, y_train, y_test = train_test_split(
    X_combined, y, test_size=0.2, stratify=y, random_state=13
)

In [43]:
scores_svm_model = run_svm(X_train, y_train, X_test, y_test)

acc: 0.7755102040816326
              precision    recall  f1-score   support

   no stance       0.78      0.80      0.79       116
   prochoice       0.73      0.71      0.72        96
     prolife       0.83      0.82      0.82        82

    accuracy                           0.78       294
   macro avg       0.78      0.78      0.78       294
weighted avg       0.78      0.78      0.78       294


[[93 16  7]
 [21 68  7]
 [ 6  9 67]]


In [44]:
cross_val(X_train, y_train)

cv scores: [0.71914894 0.78297872 0.74468085 0.71489362 0.70512821]
('avg acc:', 0.733366066557556)
std acc: 0.02803631599991725
avg f1 0.7338756339947884


# scores and subreddit?

In [45]:
X_combined = sp.hstack((X_tfidf, score, subreddits))
X_train, X_test, y_train, y_test = train_test_split(X_combined, y, test_size=0.2, random_state=13, stratify=y)

combo_svm = run_svm(X_train, y_train, X_test, y_test)

acc: 0.7244897959183674
              precision    recall  f1-score   support

   no stance       0.75      0.73      0.74       116
   prochoice       0.72      0.57      0.64        96
     prolife       0.70      0.89      0.78        82

    accuracy                           0.72       294
   macro avg       0.72      0.73      0.72       294
weighted avg       0.73      0.72      0.72       294


[[85 16 15]
 [24 55 17]
 [ 4  5 73]]


In [46]:
cross_val(X_train, y_train)

cv scores: [0.69787234 0.76595745 0.72765957 0.72765957 0.71794872]
('avg acc:', 0.7274195308237862)
std acc: 0.022126992417828794
avg f1 0.7276872453781248


# no stance?

In [47]:
labeled = labeled[labeled['my_label'] != 'no stance'].reset_index(drop=True)
labeled['my_label'].value_counts()

label_encoder = LabelEncoder()
y  = label_encoder.fit_transform(labeled['my_label'])
(dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))

{'prochoice': 0, 'prolife': 1}

In [48]:
X = tfidf.fit_transform(labeled['body'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=13
)

run_svm(X_train, y_train, X_test, y_test)

acc: 0.8707865168539326
              precision    recall  f1-score   support

   prochoice       0.82      0.97      0.89        96
     prolife       0.95      0.76      0.84        82

    accuracy                           0.87       178
   macro avg       0.89      0.86      0.87       178
weighted avg       0.88      0.87      0.87       178


[[93  3]
 [20 62]]


SVC(class_weight='balanced', kernel='linear', probability=True)

In [49]:
subreddit_code = encoder.fit_transform(labeled[['subreddit']]).toarray()

subreddits = pd.DataFrame(subreddit_code, columns=encoder.get_feature_names_out(["subreddit"]))

In [50]:
# w/ subreddit
X_combined = sp.hstack((X, subreddits))

X_train, X_test, y_train, y_test = train_test_split(
    X_combined, y, test_size=0.2, stratify=y, random_state=13
)

run_svm(X_train, y_train, X_test, y_test)

acc: 0.848314606741573
              precision    recall  f1-score   support

   prochoice       0.92      0.79      0.85        96
     prolife       0.79      0.91      0.85        82

    accuracy                           0.85       178
   macro avg       0.85      0.85      0.85       178
weighted avg       0.86      0.85      0.85       178


[[76 20]
 [ 7 75]]


SVC(class_weight='balanced', kernel='linear', probability=True)

# new labeled

In [53]:
labeled = pd.read_csv('/Users/emma/Desktop/test.csv')
labeled = labeled[labeled['my_label'] != 'unclear']
labeled['my_label'].value_counts()

KeyError: 'my_label'

In [ ]:
labeled['body'] = labeled['body'].str.replace(f"[{string.punctuation}]", " ", regex=True).str.strip()

In [ ]:
label_encoder = LabelEncoder()
y  = label_encoder.fit_transform(labeled['my_label'])
(dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))

{'no stance': 0, 'prochoice': 1, 'prolife': 2}

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

# tf idf
tfidf = TfidfVectorizer(max_features=5000, 
                        ngram_range = (1, 2), 
                        stop_words='english')
X  = tfidf.fit_transform(labeled['body'])

# train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=13
)

def run_svm(X_train, y_train, X_test, y_test):
    
    svm_model = SVC(kernel="linear", 
                    C=1.0, 
                    probability=True, 
                    class_weight="balanced")

    svm_model.fit(X_train, y_train)

    y_pred = svm_model.predict(X_test)

    print('acc:', accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))
    print("")
    print(confusion_matrix(y_test, y_pred))

    return svm_model

grid_svm = run_svm(X_train, y_train, X_test, y_test)

acc: 0.648
              precision    recall  f1-score   support

   no stance       0.54      0.61      0.57        74
   prochoice       0.64      0.64      0.64        95
     prolife       0.79      0.69      0.74        81

    accuracy                           0.65       250
   macro avg       0.66      0.65      0.65       250
weighted avg       0.66      0.65      0.65       250


[[45 21  8]
 [27 61  7]
 [12 13 56]]
